# v12 RAM signal — step-by-step debug

**The question.** On v11 a storage reorder returns a real percentage and the cube's measured
RAM moves. On v12, in the parity runs, `update_storage_dimension_order` returned `0.0` for
every reorder and `cube_memory_used` never moved — so OptimusPy had nothing to choose a
winner with and picked one by tie-break.

OptimusPy reads absolute RAM **once** (the baseline) and derives every later permutation from
the percentage the reorder returns (`CONTEXT.md`, *Scope of the v12 migration*). So the
percentage is not a nice-to-have — it is the only per-permutation measurement there is.

**What each step establishes**

| Step | Question |
|---|---|
| 1 | Does the gauge read at all, and does it carry a `Timestamp`? |
| 2 | Does an *identity* reorder return a percentage, and does it wake the gauge? |
| 3 | Does a *real* reorder return a percentage? **This is the one that matters.** |
| 4 | Across several orders: does the percentage ever move? Does the gauge? |
| 5 | Do any of the *other* metrics move with order, even if `cube_memory_used` does not? |
| 6 | After a reorder, does a **data change** make the gauge report an order-appropriate value? |

Step 6 is the discriminator. If the value only moves once data changes, the reorder *did*
change the cube and the server simply never invalidated the figure. That is a precise bug.
If it reports the same number whatever the order, then order genuinely does not affect the
v12 footprint — a different conclusion entirely.

---

> Cell **2** builds the parity fixture (`OptimusPy_Parity_Test`) so your numbers line up
> with the reference figures there. Point `CUBE` at your own cube instead if you prefer.
>
> ⚠️ **This notebook reorders a real cube.** Cell 3 captures the original storage order and
> the **Restore** cell near the bottom puts it back. Run it against the parity fixture, or a
> cube you are willing to have reordered — a reorder is a blocking server-side rebuild with
> no safe abort. Step 6 also writes one cell value (and writes it back); it is behind a flag
> that is off by default.
>
> Servers are on Tailscale addresses, so run Jupyter **with the sandbox disabled**, using the
> pyenv interpreter that has TM1py installed.

## 0 · Settings

In [12]:
# Edit these, then run every cell top to bottom. Re-run any single step as often as you like.
CONFIG_PATH = "../config/config.ini"
INSTANCE    = "tm1srv02"                  # tm1srv02 = v12, tm1srv01 = v11 (run both, compare)
CUBE        = "OptimusPy_Parity_Test"   # the parity fixture; cell 2b can build it for you

# Fixture control (cell 2b). Leave both False to use whatever is already on the server.
BUILD_FIXTURE = True    # build it if missing
FORCE_REBUILD = True    # tear down and rebuild even if present -- see the warning in 2b

# Step 6 writes one cell value and writes it back. Off by default.
ALLOW_DATA_WRITE = False

# How long to watch the gauge after a reorder before giving up on it moving.
POLL_SECONDS = 180
POLL_EVERY   = 15

import os, sys, json, time, itertools
from datetime import datetime

# Run from the repo root so CONFIG_PATH resolves; adjust if your kernel starts elsewhere.
REPO = os.path.abspath(os.getcwd())
if os.path.basename(REPO) != "optimus-py" and os.path.isdir(os.path.join(REPO, "..", "optimus-py")):
    REPO = os.path.abspath(os.path.join(REPO, ".."))
sys.path.insert(0, os.path.join(REPO, "src"))
print("repo   :", REPO)
print("config :", os.path.join(REPO, CONFIG_PATH), "exists:", os.path.exists(os.path.join(REPO, CONFIG_PATH)))

STATE = globals().get("STATE", {})   # survives re-runs of later cells

repo   : /Users/nicolasbisurgi/Work/Projects/Products/optimus-py/samples
config : /Users/nicolasbisurgi/Work/Projects/Products/optimus-py/samples/../config/config.ini exists: True


## 1 · Connect

In [6]:
from TM1py import TM1Service
from optimuspy.core import get_tm1_config
from optimuspy.metrics import detect_is_v12, unit_to_bytes, CUBE_MEMORY_METRIC

cfg = get_tm1_config(os.path.join(REPO, CONFIG_PATH))
args = dict(cfg[INSTANCE]); args["session_context"] = "optimuspy-debug-nb"

tm1 = TM1Service(**args)
version = tm1.server.get_product_version()
IS_V12  = detect_is_v12(tm1)
STATE.update(instance=INSTANCE, version=version, is_v12=IS_V12)

print(f"connected  : {INSTANCE}")
print(f"version    : {version}   -> is_v12={IS_V12}")
print(f"cube exists: {tm1.cubes.exists(CUBE)}")

connected  : tm1srv02
version    : 12.6.4   -> is_v12=True
cube exists: False


## 2 · The fixture cube — build it so the numbers compare

The reference figures below come from `OptimusPy_Parity_Test`, the cube
`samples/validate_v11_v12_parity.py` builds. This cell calls **that script's own builders**
rather than a copy, so the data is byte-identical to the runs that produced them: 300,000
cells written server-side by a MINSTD LCG on a fixed seed, no `RAND()`, so v11 and v12
receive exactly the same data.

**Shape:** 7 sparse numeric dimensions + a numeric measure dimension last (8 in all).

| Dimension | Leaves | | Dimension | Leaves |
|---|---|---|---|---|
| `_Dim1` | 200 | | `_Dim5` | 50 |
| `_Dim2` | 160 | | `_Dim6` | **25** |
| `_Dim3` | 120 | | `_Dim7` | **12** |
| `_Dim4` | 80 | | `_Measure` | 3 |

`Dim6`/`Dim7` are the pair the two versions disagreed on — cardinality ratio 2.08, inside
`TAU_RAM = 4`, so the greedy is meant to settle them **by measurement** rather than by theory.
That is exactly the decision v12 had no signal for. Step 3 swaps that pair on purpose.

### Reference numbers to compare against

| | v11 (tm1srv01) | v12 (tm1srv02) |
|---|---|---|
| Original-order RAM | **67,105,792 B** | **67,145,728 B** *(0.060% apart — they agree)* |
| `pct` returned per reorder | real, varying | **`0.0` every time** |
| Distinct RAM values over 15 permutations | **7** | **1** |
| Span | 35.9 MB – 57.5 MB | no span |
| Cold-start gauge | n/a | **40,960 B** (empty skeleton), frozen 12m23s |
| Greedy winner | `Dim6, Dim7` | `Dim7, Dim6` (tie-break, not a measurement) |

*(These are from the agent's live runs, not this notebook — they are what you are trying to
reproduce, not something this notebook has verified.)*

> ⚠️ **On rebuilding.** The builders use `update_or_create`, so a fixture left behind by a
> crashed run is silently **adopted** — including one left in a *reordered* storage order by a
> previous run, which would quietly become your "original order". That already happened once
> here. If a cube is already present and you did not build it in this session, prefer
> `FORCE_REBUILD = True`.

In [13]:
# Import the parity script itself so the fixture is built by the same code, not a copy.
sys.path.insert(0, os.path.join(REPO, "samples"))
import importlib
parity = importlib.import_module("validate_v11_v12_parity")

print(f"fixture cube   : {parity.CUBE}")
print(f"dimensions     : {parity._dimension_names()}")
print(f"leaf counts    : {parity.DIM_SIZES}   measures={parity.MEASURES}")
print(f"fill           : {parity.FILL_CELLS:,} cells, LCG seed {parity.SEED}")
print()

exists = tm1.cubes.exists(parity.CUBE)
print(f"already on {INSTANCE}: {exists}")
if exists:
    print(f"  its storage order : {tm1.cubes.get_storage_dimension_order(parity.CUBE)}")
    print(f"  built order       : {parity._dimension_names()}")
    if tm1.cubes.get_storage_dimension_order(parity.CUBE) != parity._dimension_names():
        print("  >>> NOT at the built order. A previous run left it reordered — rebuild, or")
        print("  >>> whatever you measure as the 'original order' is that run's leftover.")

if FORCE_REBUILD and exists:
    print("\ntearing down first (FORCE_REBUILD)...")
    parity.teardown_instance(tm1)
    exists = False

if BUILD_FIXTURE and not exists:
    print("\nbuilding — dimensions, cube, then a server-side TI fill of 300,000 cells...")
    t0 = time.time()
    parity.setup_instance(tm1)
    print(f"built in {time.time() - t0:.1f}s")
    print(f"  storage order : {tm1.cubes.get_storage_dimension_order(parity.CUBE)}")
    # A fresh fixture is the cold-start case: on v12 the gauge can sit at the ~40 KB
    # skeleton here for many minutes. That is Step 1's first reading, and it is a finding,
    # not a reason to wait before starting.
    b, raw, unit, ts = (None, None, None, None)
    rows = tm1.metrics.by_cube(cube=parity.CUBE)
    r = next((x for x in rows if x.get("Metric") == "cube_memory_used"), None)
    if r:
        print(f"  gauge right after the load : {r.get('Value')} {r.get('Unit')}   ts={r.get('Timestamp')}")
elif not BUILD_FIXTURE and not exists:
    print("\nnot built — set BUILD_FIXTURE = True to create it")
elif exists:
    print("\nusing the fixture already on the server (set FORCE_REBUILD = True for a clean one)")

CUBE = parity.CUBE
STATE["fixture"] = {"cube": CUBE, "built_order": parity._dimension_names()}

fixture cube   : OptimusPy_Parity_Test
dimensions     : ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Measure']
leaf counts    : {'OptimusPy_Parity_Test_Dim1': 200, 'OptimusPy_Parity_Test_Dim2': 160, 'OptimusPy_Parity_Test_Dim3': 120, 'OptimusPy_Parity_Test_Dim4': 80, 'OptimusPy_Parity_Test_Dim5': 50, 'OptimusPy_Parity_Test_Dim6': 25, 'OptimusPy_Parity_Test_Dim7': 12}   measures=['Value', 'Count', 'Amount']
fill           : 300,000 cells, LCG seed 20260605

already on tm1srv02: False

building — dimensions, cube, then a server-side TI fill of 300,000 cells...


KeyboardInterrupt: 

## 3 · The cube, and the original order (captured for restore)

In [14]:
storage      = tm1.cubes.get_storage_dimension_order(CUBE)
presentation = tm1.cubes.get_dimension_names(CUBE)

# Capture the original ONCE per session — re-running this cell must not overwrite it with a
# reordered state, or the restore cell would put back the wrong thing.
if STATE.get("original_order_cube") != CUBE:
    STATE["original_order"] = list(storage)
    STATE["original_order_cube"] = CUBE
    print(f"captured original storage order of '{CUBE}' for restore")

print(f"\nstorage order      : {storage}")
print(f"presentation order : {presentation}")
print(f"same?              : {storage == presentation}")
print(f"ORIGINAL (restore) : {STATE['original_order']}")

print("\nleaf counts:")
for d in storage:
    hier = "Leaves" if tm1.hierarchies.exists(dimension_name=d, hierarchy_name="Leaves") else d
    n = len(tm1.elements.get_leaf_element_names(d, hier))
    print(f"  {d:<40} {n:>7}")

captured original storage order of 'OptimusPy_Parity_Test' for restore

storage order      : ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Measure']
presentation order : ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Measure']
same?              : True
ORIGINAL (restore) : ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Measure']

leaf counts:
  OptimusPy_Parity_Test_Dim1                   200
  OptimusPy_Parity_Test_Dim2              

## 4 · Every metric the server reports for this cube

Nobody has looked at the other metrics. If `cube_memory_used` is flat but something else
moves with order, this is a metric-*selection* problem on our side, not a platform limit.
Note whether rows carry a `Timestamp` — on v12 that separates *stale* from
*fresh-and-genuinely-unchanged*.

In [15]:
rows = tm1.metrics.by_cube(cube=CUBE)
print(f"{len(rows)} rows\n")
keys = sorted({k for r in rows for k in r})
print("keys present:", keys, "\n")
for r in rows:
    print(json.dumps(r, indent=2, default=str))

6 rows

keys present: ['Category', 'CubeName', 'DatabaseID', 'DatabaseName', 'Metric', 'NativeName', 'ReplicaID', 'TimeInterval', 'Timestamp', 'Unit', 'Value'] 

{
  "Category": "by_cube",
  "CubeName": "OptimusPy_Parity_Test",
  "Metric": "cube_memory_used",
  "NativeName": "cube_memory_used",
  "Value": 40,
  "Unit": "KB",
  "ReplicaID": 0,
  "TimeInterval": "LATEST",
  "Timestamp": "2026-09-22T13:29:49.000Z",
  "DatabaseName": "cw-v12-test",
  "DatabaseID": "<DATABASE-ID>"
}
{
  "Category": "by_cube",
  "CubeName": "OptimusPy_Parity_Test",
  "Metric": "cube_memory_used_for_cell_values",
  "NativeName": "cube_memory_used_for_cell_values",
  "Value": 40960,
  "Unit": "B",
  "ReplicaID": 0,
  "TimeInterval": "LATEST",
  "Timestamp": "2026-09-22T13:29:49.000Z",
  "DatabaseName": "cw-v12-test",
  "DatabaseID": "<DATABASE-ID>"
}
{
  "Category": "by_cube",
  "CubeName": "OptimusPy_Parity_Test",
  "Metric": "cube_memory_used_for_feeder_flags",
  "NativeName": "cube_memory_used_for_feeder_fl

## 5 · Helpers

In [16]:
from TM1py.Utils import format_url

def read_rows():
    return tm1.metrics.by_cube(cube=CUBE)

def gauge(rows=None):
    """cube_memory_used as (bytes, raw_value, unit, timestamp) — no settling, one read."""
    rows = read_rows() if rows is None else rows
    r = next((x for x in rows if x.get("Metric") == CUBE_MEMORY_METRIC), None)
    if r is None or r.get("Value") is None:
        return (None, None, None, None)
    return (unit_to_bytes(r["Value"], r.get("Unit")), r["Value"], r.get("Unit"),
            r.get("Timestamp"))

def poll_gauge(seconds=POLL_SECONDS, every=POLL_EVERY, label=""):
    """Watch the gauge and print every sample. Returns the list of (elapsed, bytes, ts)."""
    t0, seen = time.time(), []
    while True:
        b, raw, unit, ts = gauge()
        el = round(time.time() - t0)
        seen.append((el, b, ts))
        print(f"  [{label}] t+{el:>4}s  {('%15s' % f'{b:,.0f}') if b is not None else '           None'} B"
              f"   (raw={raw} {unit})   ts={ts}")
        if el >= seconds:
            break
        time.sleep(every)
    distinct = sorted({s[1] for s in seen if s[1] is not None})
    print(f"  [{label}] distinct values seen: {len(distinct)} -> {[f'{d:,.0f}' for d in distinct]}")
    return seen

def reorder_raw(order):
    """Reorder via the raw REST call, so we see EXACTLY what the server returns.

    TM1py parses response.json()['value'] as the percent change. If v12 answers with a
    different shape, that parse yields 0.0 and the defect is client-side, not server-side.
    This is the only way to tell the two apart.
    """
    url = format_url("/Cubes('{}')/tm1.ReorderDimensions", CUBE)
    payload = {"Dimensions@odata.bind": [format_url("Dimensions('{}')", d) for d in order]}
    t0 = time.time()
    resp = tm1.cubes._rest.POST(url=url, data=json.dumps(payload))
    secs = round(time.time() - t0, 2)
    try:
        body = resp.json()
    except Exception as e:
        body = f"<not json: {e}>"
    return {"status": resp.status_code, "seconds": secs, "body": body,
            "text": (resp.text or "")[:800]}

def show_reorder(res):
    print(f"  HTTP {res['status']}   {res['seconds']}s")
    print(f"  raw body : {res['body']!r}")
    if isinstance(res["body"], dict):
        print(f"  keys     : {list(res['body'].keys())}")
        print(f"  ['value']: {res['body'].get('value')!r}   <- this is what TM1py returns as the %")
    else:
        print(f"  raw text : {res['text']!r}")

def fmt(b):
    return "None" if b is None else f"{b:,.0f}"

print("helpers ready")

helpers ready


## Step 1 · Does the gauge read at all?

Baseline, before anything is touched. On v12 this is the value that can sit at the empty-cube
skeleton (~40 KB) long after the data is loaded.

In [17]:
b, raw, unit, ts = gauge()
STATE["baseline_first_read"] = b
print(f"first read: {fmt(b)} B   (raw={raw} {unit})   ts={ts}")
print()
print("watching it (Ctrl-C to stop early):")
STATE["baseline_trajectory"] = poll_gauge(seconds=POLL_SECONDS, every=POLL_EVERY, label="baseline")

first read: 40,960 B   (raw=40 KB)   ts=2026-09-22T13:29:49.000Z

watching it (Ctrl-C to stop early):
  [baseline] t+   0s           40,960 B   (raw=40 KB)   ts=2026-09-22T13:29:49.000Z
  [baseline] t+  15s           40,960 B   (raw=40 KB)   ts=2026-09-22T13:29:49.000Z
  [baseline] t+  31s           40,960 B   (raw=40 KB)   ts=2026-09-22T13:29:49.000Z
  [baseline] t+  46s           40,960 B   (raw=40 KB)   ts=2026-09-22T13:29:49.000Z


KeyboardInterrupt: 

## Step 2 · Identity reorder

Reorder the cube to the order it is *already in*. Nothing changes physically, so a truthful
percentage is `0.0` — that is expected here and is **not** the defect. What this step is for:

- does the call succeed and return a well-formed body?
- does it wake a frozen gauge? (the earlier run saw a frozen gauge come alive ~15s after a
  reorder, which is why OptimusPy's own first step — the original-order measurement — is an
  identity reorder followed by an absolute read)

In [18]:
order = tm1.cubes.get_storage_dimension_order(CUBE)
print(f"applying (identity): {order}\n")
res = reorder_raw(order)
show_reorder(res)
STATE["identity_reorder"] = res

print("\ngauge immediately after:")
b, raw, unit, ts = gauge(); print(f"  {fmt(b)} B  (raw={raw} {unit})  ts={ts}")
print("\nthen:")
STATE["identity_trajectory"] = poll_gauge(seconds=POLL_SECONDS, every=POLL_EVERY, label="identity")

applying (identity): ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Measure']

  HTTP 201   0.56s
  raw body : {'@odata.context': '../$metadata#Edm.Double', 'value': 0}
  keys     : ['@odata.context', 'value']
  ['value']: 0   <- this is what TM1py returns as the %

gauge immediately after:
  67,145,728 B  (raw=65572 KB)  ts=2026-09-22T13:34:31.000Z

then:
  [identity] t+   0s       67,145,728 B   (raw=65572 KB)   ts=2026-09-22T13:34:31.000Z
  [identity] t+  15s       67,145,728 B   (raw=65572 KB)   ts=2026-09-22T13:34:31.000Z
  [identity] t+  31s       67,145,728 B   (raw=65572 KB)   ts=2026-09-22T13:34:31.000Z
  [identity] t+  46s       67,145,728 B   (raw=65572 KB)   ts=2026-09-22T13:34:31.000Z
  [identity] t+  61s       67,145,728 B   (raw=65572 KB)   ts=2026-09-22T13:34:31.000Z


KeyboardInterrupt: 

## Step 3 · One real reorder — **the step that matters**

A genuinely different order. On v11 this returns a non-zero percentage. The parity runs saw
`0.0` here on v12, every time, for a reorder that `get_storage_dimension_order` confirms
landed.

Read the raw body carefully. `{"value": 0}` means the server said zero. Anything else —
a different key, a different shape, an absolute size instead of a percent — means TM1py 2.3.0
is parsing a v11-shaped answer and the defect is on our side of the wire.

In [ ]:
current = tm1.cubes.get_storage_dimension_order(CUBE)
# Swap the two *smallest* dimensions: a real change, cheap to rebuild, and the pair the
# parity gate actually disagreed on.
candidate = list(current)
candidate[-3], candidate[-2] = candidate[-2], candidate[-3]

print(f"from : {current}")
print(f"to   : {candidate}")
print(f"changed? {current != candidate}\n")

before_b, *_ = gauge()
res = reorder_raw(candidate)
show_reorder(res)
STATE["real_reorder"] = res

landed = tm1.cubes.get_storage_dimension_order(CUBE)
print(f"\n  order after the call : {landed}")
print(f"  reorder landed?      : {landed == candidate}")

after_b, raw, unit, ts = gauge()
print(f"\n  gauge before : {fmt(before_b)} B")
print(f"  gauge after  : {fmt(after_b)} B   ts={ts}")
print(f"  moved?       : {before_b != after_b}")
print("\nthen:")
STATE["real_trajectory"] = poll_gauge(seconds=POLL_SECONDS, every=POLL_EVERY, label="real")

from : ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Measure']
to   : ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Measure']
changed? True

  HTTP 201   0.94s
  raw body : {'@odata.context': '../$metadata#Edm.Double', 'value': 0}
  keys     : ['@odata.context', 'value']
  ['value']: 0   <- this is what TM1py returns as the %

  order after the call : ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Measure']
  reorder landed?      : T

## Step 3b · The raw call, to paste into another client

`update_storage_dimension_order` is four lines of TM1py:

```python
url = format_url("/Cubes('{}')/tm1.ReorderDimensions", cube_name)
payload = {"Dimensions@odata.bind": [format_url("Dimensions('{}')", d) for d in dimension_names]}
response = self._rest.POST(url=url, data=json.dumps(payload))
return response.json()["value"]          # <- the percentage
```

So the `0.0` OptimusPy sees is literally `response.json()["value"]`. This cell prints the
full URL, headers and body, and emits a `curl` you can run from anywhere.

**Two variants worth trying in your client**, because they test different things:

1. **As TM1py sends it** — `Accept: application/json;odata.metadata=none,text/plain`.
2. **With full metadata** — `Accept: application/json;odata.metadata=minimal`. OData omits
   control information under `metadata=none`. `value` is the payload key rather than an
   annotation so it *should* survive, but if variant 2 returns a real percentage and
   variant 1 returns `0`, the defect is in the request we send, not in the server.

The action itself is working — `get_storage_dimension_order` confirms the new order landed.
It is only the **return payload** that is empty of information, which is why the raw body
matters more than the status code.

In [20]:
SHOW_AUTH = False    # True embeds a working credential in this notebook's output — see below

rest = tm1.cubes._rest
base = rest._base_url
path = f"/Cubes('{CUBE}')/tm1.ReorderDimensions"
full = base + path

current = tm1.cubes.get_storage_dimension_order(CUBE)
candidate = list(current)
candidate[-3], candidate[-2] = candidate[-2], candidate[-3]   # swap the two smallest dims
body = {"Dimensions@odata.bind": [f"Dimensions('{d}')" for d in candidate]}

print("METHOD : POST")
print("URL    :", full)
print("\nBODY:")
print(json.dumps(body, indent=2))

print("\nHEADERS (as TM1py sends them):")
hdrs = dict(rest._s.headers)
for k, v in hdrs.items():
    if k.lower() == "authorization" and not SHOW_AUTH:
        v = v.split(" ")[0] + " <redacted — set SHOW_AUTH=True to print>"
    print(f"  {k}: {v}")

cookies = rest._s.cookies.get_dict()
if cookies:
    print("\nCOOKIES:")
    for k, v in cookies.items():
        print(f"  {k}: {v if SHOW_AUTH else '<redacted>'}")

print("\n" + "=" * 100)
print("curl — variant 1, exactly as TM1py sends it")
print("=" * 100)
auth_bits = []
if "Authorization" in hdrs:
    auth_bits.append(f"  -H 'Authorization: {hdrs['Authorization'] if SHOW_AUTH else '<PASTE>'}' \\")
for k, v in cookies.items():
    auth_bits.append(f"  -b '{k}={v if SHOW_AUTH else '<PASTE>'}' \\")
if not auth_bits:
    auth_bits.append("  -u '<user>:<password>' \\")

def curl(accept):
    lines = [f"curl -sS -k -X POST '{full}' \\"]
    lines += auth_bits
    lines += [f"  -H 'Content-Type: application/json' \\",
              f"  -H 'Accept: {accept}' \\",
              f"  -d '{json.dumps(body)}' \\",
              "  -D -"]          # -D - prints response headers too
    return "\n".join(lines)

print(curl(hdrs.get("Accept", "application/json;odata.metadata=none,text/plain")))
print("\n" + "=" * 100)
print("curl — variant 2, same call with full OData metadata")
print("=" * 100)
print(curl("application/json;odata.metadata=minimal"))

print("""
NOTE: running this reorders the cube. Run the Restore cell afterwards.
      SHOW_AUTH=True prints a live credential; if you save the notebook with that output
      the credential is saved with it. Clear the output before committing.""")

METHOD : POST
URL    : https://<PA-HOST>/api/<TENANT>/v0/tm1/cw-v12-test/api/v1/Cubes('OptimusPy_Parity_Test')/tm1.ReorderDimensions

BODY:
{
  "Dimensions@odata.bind": [
    "Dimensions('OptimusPy_Parity_Test_Dim2')",
    "Dimensions('OptimusPy_Parity_Test_Dim3')",
    "Dimensions('OptimusPy_Parity_Test_Dim4')",
    "Dimensions('OptimusPy_Parity_Test_Dim5')",
    "Dimensions('OptimusPy_Parity_Test_Dim6')",
    "Dimensions('OptimusPy_Parity_Test_Measure')",
    "Dimensions('OptimusPy_Parity_Test_Dim7')",
    "Dimensions('OptimusPy_Parity_Test_Dim1')"
  ]
}

HEADERS (as TM1py sends them):
  User-Agent: python-requests/2.34.2
  Accept-Encoding: gzip, deflate, zstd
  Accept: */*
  Connection: keep-alive

COOKIES:
  paSession: <redacted>

curl — variant 1, exactly as TM1py sends it
curl -sS -k -X POST 'https://<PA-HOST>/api/<TENANT>/v0/tm1/cw-v12-test/api/v1/Cubes('OptimusPy_Parity_Test')/tm1.ReorderDimensions' \
  -b 'paSession=<PASTE>' \
  -H 'Content-Type: application/json' \
  -H 'Acce

## Step 3c · What does the service say the action returns?

The OData `$metadata` document declares `tm1.ReorderDimensions`'s return type. If v11 declares
a return type and v12 declares none — or a different one — that is the contract change, in
writing, from the server itself. Run this on **both** instances and compare the two lines.

In [ ]:
meta = rest.GET("/$metadata", headers={"Accept": "application/xml"}).text
i = meta.find("ReorderDimensions")
if i < 0:
    print("ReorderDimensions not found in $metadata")
else:
    start = meta.rfind("<", 0, max(0, i - 400))
    print(meta[start:i + 900])

## Step 4 · Several orders in a row

The summary table. `pct` is what TM1py hands OptimusPy — the number every permutation's RAM
is derived from. `gauge_after` is an independent absolute read, so the two channels can be
compared. If `pct` is `0.0` on every row while the orders genuinely differ, that alone
explains every identical RAM figure in the parity CSVs.

`settle_seconds` controls how long to wait before `gauge_settled` — set it to 0 for a fast
pass, or to 60+ if you want to give a lagging gauge a chance.

In [21]:
settle_seconds = 30          # 0 = don't wait

base = tm1.cubes.get_storage_dimension_order(CUBE)
n = len(base)
# Four distinct orders built from the current one: identity, two adjacent swaps, one rotation.
orders = [
    list(base),
    [*base[:n-3], base[n-2], base[n-3], base[n-1]],
    [base[1], base[0], *base[2:]],
    [*base[1:n-1], base[0], base[n-1]],
]

results = []
for i, o in enumerate(orders):
    print(f"\n--- {i+1}/{len(orders)}  {o}")
    before_b, *_ = gauge()
    res = reorder_raw(o)
    pct = res["body"].get("value") if isinstance(res["body"], dict) else None
    landed = tm1.cubes.get_storage_dimension_order(CUBE)
    after_b, _, _, ts = gauge()
    if settle_seconds:
        time.sleep(settle_seconds)
    settled_b, _, _, ts2 = gauge()
    results.append({"order": o, "landed": landed == o, "http": res["status"],
                    "secs": res["seconds"], "pct": pct,
                    "gauge_before": before_b, "gauge_after": after_b,
                    "gauge_settled": settled_b, "ts": ts2})
    print(f"    pct={pct!r}  landed={landed == o}  gauge {fmt(before_b)} -> {fmt(after_b)} -> {fmt(settled_b)}")

STATE["sweep"] = results

print("\n" + "=" * 104)
print(f"{'#':<3}{'landed':<8}{'pct':>14}{'gauge_before':>18}{'gauge_after':>18}{'gauge_settled':>18}{'secs':>8}")
print("-" * 104)
for i, r in enumerate(results):
    print(f"{i+1:<3}{str(r['landed']):<8}{str(r['pct']):>14}{fmt(r['gauge_before']):>18}"
          f"{fmt(r['gauge_after']):>18}{fmt(r['gauge_settled']):>18}{r['secs']:>8}")
print("=" * 104)
pcts   = {r["pct"] for r in results}
gauges = {r["gauge_settled"] for r in results}
print(f"distinct pct values     : {len(pcts)}   {pcts}")
print(f"distinct settled gauges : {len(gauges)}  {[fmt(g) for g in gauges]}")
print()
if len(pcts) == 1 and next(iter(pcts)) in (0, 0.0):
    print(">>> Every reorder returned 0. This is the whole defect: OptimusPy derives every")
    print(">>> permutation's RAM from this number, so all orders come out identical and the")
    print(">>> winner is a tie-break. Compare the raw body in Step 3 against v11's.")
else:
    print(">>> The percentage moves. Whatever went wrong in the parity run was not this.")


--- 1/4  ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Measure']
    pct=0  landed=True  gauge 67,145,728 -> 67,145,728 -> 67,145,728

--- 2/4  ['OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Measure']
    pct=0  landed=True  gauge 67,145,728 -> 67,145,728 -> 67,145,728

--- 3/4  ['OptimusPy_Parity_Test_Dim2', 'OptimusPy_Parity_Test_Dim1', 'OptimusPy_Parity_Test_Dim3', 'OptimusPy_Parity_Test_Dim4', 'OptimusPy_Parity_Test_Dim5', 'OptimusPy_Parity_Test_Dim6', 'OptimusPy_Parity_Test_Dim7', 'OptimusPy_Parity_Test_Measure']
    pct=0  landed=True  gauge 67,145,728 -> 67,145,728 -> 67,145,728

--- 4/4  ['OptimusPy_Parity_Te

## Step 5 · Does **any** metric move with order?

`cube_memory_used` is the metric OptimusPy reads, but it is not the only one the server
reports. If another one tracks dimension order, the signal exists and we are reading the
wrong row.

In [ ]:
base = tm1.cubes.get_storage_dimension_order(CUBE)
n = len(base)
probe_orders = [list(base), [base[1], base[0], *base[2:]]]

matrix = {}
for o in probe_orders:
    reorder_raw(o)
    time.sleep(5)
    for r in read_rows():
        m = r.get("Metric")
        if m:
            matrix.setdefault(m, []).append(r.get("Value"))

print(f"{'metric':<42}{'values across orders':<40}{'moves?'}")
print("-" * 95)
movers = []
for m, vals in sorted(matrix.items()):
    moves = len(set(map(str, vals))) > 1
    if moves:
        movers.append(m)
    print(f"{m:<42}{str(vals):<40}{'YES' if moves else '-'}")
print()
print(f">>> metrics that move with dimension order: {movers or 'NONE'}")
STATE["metric_matrix"] = matrix

## Step 6 · The discriminator — reorder, then change data

If the gauge only moves once **data** changes, the reorder really did change the cube and the
server simply never invalidated the figure (stale, not coarse, not "order doesn't matter").
That is a precise, reportable bug.

Needs `ALLOW_DATA_WRITE = True` in the settings cell. It writes one cell and writes the
previous value back.

In [ ]:
if not ALLOW_DATA_WRITE:
    print("skipped — set ALLOW_DATA_WRITE = True in the settings cell to run this step")
else:
    dims = tm1.cubes.get_dimension_names(CUBE)
    coords = []
    for d in dims:
        hier = "Leaves" if tm1.hierarchies.exists(dimension_name=d, hierarchy_name="Leaves") else d
        coords.append(tm1.elements.get_leaf_element_names(d, hier)[0])
    print(f"cell: {coords}")

    before_b, *_ = gauge()
    old = tm1.cells.get_value(CUBE, coords)
    print(f"gauge before write : {fmt(before_b)} B   (cell currently {old!r})")

    tm1.cells.write_value((old or 0) + 12345, CUBE, coords)
    print("wrote a new value; watching the gauge:")
    traj = poll_gauge(seconds=POLL_SECONDS, every=POLL_EVERY, label="after-write")

    tm1.cells.write_value(old or 0, CUBE, coords)
    print(f"\nrestored the cell to {old!r}")

    after_b, *_ = gauge()
    STATE["discriminator"] = {"before": before_b, "after": after_b, "trajectory": traj}
    print(f"\ngauge before write : {fmt(before_b)} B")
    print(f"gauge after  write : {fmt(after_b)} B")
    print()
    if before_b is not None and after_b is not None and before_b != after_b:
        print(">>> The gauge moved on a DATA change while it had not moved on a REORDER.")
        print(">>> Reading: the reorder does change the cube, and the server is not")
        print(">>> invalidating cube_memory_used for it. Stale, not coarse.")
    else:
        print(">>> The gauge did not move on a data change either — the gauge itself is the")
        print(">>> problem, independently of dimension order. Note how long it stays frozen.")

---

# Step 7 · Side by side — same cube, same changes, two versions

**Why this cell exists.** Step 4 showed v12 returning `0` for some reorders and
`-12.493137314707493` for another, with the gauge moving to match **to the byte** in 1.3s:

```
67,145,728 x (1 - 0.12493137314707493) = 58,757,120     <- exactly what the gauge then read
```

So v12's percentage channel works and its gauge is not frozen. A `0` means *this reorder did
not change the footprint* — which for an adjacent swap of the two smallest dimensions is very
likely the truth. That is not a defect, and it is not what made the parity gate fail.

**The question that is actually open:** the parity greedy saw **7 distinct RAM values on v11
and 1 on v12** over the same 15 permutations. Either v11 and v12 genuinely disagree about
which rearrangements cost memory, or the two runs were not comparing the same thing.

This cell settles it the only way that can't be argued with: the **same starting order**, the
**same sequence of changes**, applied to **both servers**, percentages and RAM side by side.

- A row where **one returns 0 and the other doesn't** is a real disagreement — the thing to
  take to IBM.
- Rows where **both return 0** are changes that genuinely don't matter on either engine.
- Rows where **both return the same percentage** mean the engines agree and the parity
  failure is somewhere else entirely — in which case look at *which orders the greedy chose*,
  not at the server.

RAM is compared as a **ratio to each server's own canonical baseline**, since the two
baselines differ slightly (67,105,792 vs 67,145,728 — 0.06% apart) for reasons that have
nothing to do with dimension order.

In [22]:
# --- which two instances to compare -----------------------------------------
LEFT  = "tm1srv01"     # v11
RIGHT = "tm1srv02"     # v12 (use tm1srv02 for the newer one)

SETTLE_SECONDS = 10    # pause before the "after" gauge read; raise if a gauge lags
RESTORE_AFTER  = True  # put both cubes back to the canonical order when finished

from optimuspy.metrics import ram_source_ready

def _connect(name):
    a = dict(cfg[name]); a["session_context"] = "optimuspy-debug-nb"
    return TM1Service(**a)

def _gauge(tm1):
    rows = tm1.metrics.by_cube(cube=CUBE)
    r = next((x for x in rows if x.get("Metric") == CUBE_MEMORY_METRIC), None)
    if not r or r.get("Value") is None:
        return None
    return unit_to_bytes(r["Value"], r.get("Unit"))

def _reorder(tm1, order):
    """Raw call so we see the server's own answer, not TM1py's parse of it."""
    url = format_url("/Cubes('{}')/tm1.ReorderDimensions", CUBE)
    payload = {"Dimensions@odata.bind": [format_url("Dimensions('{}')", d) for d in order]}
    resp = tm1.cubes._rest.POST(url=url, data=json.dumps(payload))
    body = resp.json()
    return body.get("value") if isinstance(body, dict) else body

left, right = _connect(LEFT), _connect(RIGHT)
print(f"{LEFT:<10} {left.server.get_product_version()}")
print(f"{RIGHT:<10} {right.server.get_product_version()}")

# --- the canonical order and the changes, identical for both -----------------
canonical = parity._dimension_names()          # Dim1..Dim7 (200,160,120,80,50,25,12) + Measure
d = canonical
CHANGES = [
    ("canonical (identity)",            list(d)),
    ("swap the 2 smallest (D6<->D7)",   [*d[:5], d[6], d[5], d[7]]),
    ("swap the 2 largest (D1<->D2)",    [d[1], d[0], *d[2:]]),
    ("largest D1 -> position 6",        [*d[1:7], d[0], d[7]]),
    ("smallest D7 -> position 0",       [d[6], *d[:6], d[7]]),
    ("reverse the 7 sparse dims",       [*reversed(d[:7]), d[7]]),
]

# --- both servers start from the canonical order -----------------------------
for name, tm1 in ((LEFT, left), (RIGHT, right)):
    if tm1.cubes.get_storage_dimension_order(CUBE) != canonical:
        _reorder(tm1, canonical)
    print(f"{name}: at canonical order = {tm1.cubes.get_storage_dimension_order(CUBE) == canonical}")

with ram_source_ready(left, False):      # v11 needs the Performance Monitor on to report
    base_l = _gauge(left)
base_r = _gauge(right)
print(f"\nbaseline {LEFT:<10} {format(base_l, ',.0f') if base_l else None} B")
print(f"baseline {RIGHT:<10} {format(base_r, ',.0f') if base_r else None} B")

# --- apply the same change to both, one at a time ----------------------------
rows = []
for label, order in CHANGES:
    rec = {"change": label}
    for side, name, tm1, base in (("l", LEFT, left, base_l), ("r", RIGHT, right, base_r)):
        if side == "l":
            with ram_source_ready(tm1, False):
                pct = _reorder(tm1, order)
                time.sleep(SETTLE_SECONDS)
                ram = _gauge(tm1)
        else:
            pct = _reorder(tm1, order)
            time.sleep(SETTLE_SECONDS)
            ram = _gauge(tm1)
        rec[f"{side}_pct"]   = pct
        rec[f"{side}_ram"]   = ram
        rec[f"{side}_ratio"] = (ram / base) if (ram and base) else None
        rec[f"{side}_landed"] = tm1.cubes.get_storage_dimension_order(CUBE) == order
    rows.append(rec)
    print(f"  {label:<32} {LEFT} pct={rec['l_pct']!s:<22} {RIGHT} pct={rec['r_pct']!s}")

# --- the table ---------------------------------------------------------------
print("\n" + "=" * 118)
print(f"{'change':<32}{LEFT+' pct':>16}{LEFT+' x base':>14}{RIGHT+' pct':>16}{RIGHT+' x base':>14}{'verdict':>24}")
print("-" * 118)
disagreements = []
for r in rows:
    lp, rp = r["l_pct"], r["r_pct"]
    lz, rz = (lp in (0, 0.0)), (rp in (0, 0.0))
    if lz and rz:
        verdict = "both 0 — no cost"
    elif lz != rz:
        verdict = ">>> DISAGREE <<<"
        disagreements.append(r)
    elif lp is not None and rp is not None and abs(lp - rp) <= 0.5:
        verdict = "agree"
    else:
        verdict = "differ in size"
    lr = f"{r['l_ratio']:.4f}" if r["l_ratio"] else "-"
    rr = f"{r['r_ratio']:.4f}" if r["r_ratio"] else "-"
    print(f"{r['change']:<32}{str(round(lp, 4) if lp is not None else None):>16}{lr:>14}"
          f"{str(round(rp, 4) if rp is not None else None):>16}{rr:>14}{verdict:>24}")
print("=" * 118)

print(f"\ndistinct {LEFT} percentages : {sorted({str(round(r['l_pct'],4)) for r in rows if r['l_pct'] is not None})}")
print(f"distinct {RIGHT} percentages : {sorted({str(round(r['r_pct'],4)) for r in rows if r['r_pct'] is not None})}")
print()
if disagreements:
    print(">>> Real disagreement on these changes — one engine charges for them, the other")
    print(">>> does not. This is the repro to take to IBM:")
    for r in disagreements:
        print(f"      {r['change']}: {LEFT}={r['l_pct']}  {RIGHT}={r['r_pct']}")
else:
    print(">>> No disagreement. Both engines charge for the same rearrangements, so the")
    print(">>> parity failure is NOT the reorder API. Next place to look: which orders the")
    print(">>> greedy actually generated on each side, and whether the v12 run started")
    print(">>> against a cold (skeleton) baseline.")

STATE["side_by_side"] = rows

if RESTORE_AFTER:
    for name, tm1 in ((LEFT, left), (RIGHT, right)):
        _reorder(tm1, canonical)
        print(f"\nrestored {name} to canonical: {tm1.cubes.get_storage_dimension_order(CUBE) == canonical}")
left.logout(); right.logout()

tm1srv01   11.8.02200.2
tm1srv02   12.6.4
tm1srv01: at canonical order = True
tm1srv02: at canonical order = True

baseline tm1srv01   67,105,792 B
baseline tm1srv02   67,145,728 B
  canonical (identity)             tm1srv01 pct=0                      tm1srv02 pct=0
  swap the 2 smallest (D6<->D7)    tm1srv01 pct=0                      tm1srv02 pct=0
  swap the 2 largest (D1<->D2)     tm1srv01 pct=0                      tm1srv02 pct=0
  largest D1 -> position 6         tm1srv01 pct=-12.50038148136845     tm1srv02 pct=-12.493137314707493
  smallest D7 -> position 0        tm1srv01 pct=0                      tm1srv02 pct=0
  reverse the 7 sparse dims        tm1srv01 pct=-14.286212549265809    tm1srv02 pct=-14.27675148135239

change                              tm1srv01 pcttm1srv01 x base    tm1srv02 pcttm1srv02 x base                 verdict
----------------------------------------------------------------------------------------------------------------------
canonical (identity)         

## Restore · put the cube back the way it was

In [ ]:
orig = STATE.get("original_order")
if not orig or STATE.get("original_order_cube") != CUBE:
    print(f"no original order captured for '{CUBE}' — run section 3 first")
    orig = None
if not orig:
    pass
else:
    now = tm1.cubes.get_storage_dimension_order(CUBE)
    if now == orig:
        print(f"already at the original order: {orig}")
    else:
        print(f"from : {now}")
        print(f"to   : {orig}")
        res = reorder_raw(orig)
        show_reorder(res)
        print(f"\nverified: {tm1.cubes.get_storage_dimension_order(CUBE) == orig}")

## Teardown · remove the fixture (optional)

Only if you built it here. **Leaving it behind is the trap**: the next run adopts it, in
whatever storage order this session left it in, and reports green.

In [ ]:
DROP_FIXTURE = False    # flip to True to delete the cube and its 8 dimensions

if not DROP_FIXTURE:
    print("skipped — set DROP_FIXTURE = True to remove the fixture")
    if tm1.cubes.exists(CUBE):
        now = tm1.cubes.get_storage_dimension_order(CUBE)
        built = STATE.get("fixture", {}).get("built_order")
        print(f"\nleaving '{CUBE}' on {INSTANCE}")
        print(f"  storage order now : {now}")
        if built and now != built:
            print("  >>> NOT at the built order. Either run the Restore cell above, or drop it —")
            print("  >>> a later run will adopt this order as its 'original'.")
else:
    parity.teardown_instance(tm1)
    print(f"dropped '{CUBE}' and its dimensions from {INSTANCE}")
    print(f"cube exists now: {tm1.cubes.exists(CUBE)}")

## Summary — what to carry back

Run this on **tm1srv01 (v11) and tm1srv02 (v12)** and put the two outputs side by side.
The v11 column is the control: it is what a working answer looks like on byte-identical data.

In [ ]:
print(f"instance : {STATE.get('instance')}   version {STATE.get('version')}   is_v12={STATE.get('is_v12')}")
print(f"cube     : {CUBE}")
print()
print(f"baseline first read      : {fmt(STATE.get('baseline_first_read'))} B")
sw = STATE.get("sweep") or []
if sw:
    print(f"reorders performed       : {len(sw)}   all landed: {all(r['landed'] for r in sw)}")
    print(f"distinct pct returned    : {sorted({str(r['pct']) for r in sw})}")
    print(f"distinct settled gauges  : {sorted({fmt(r['gauge_settled']) for r in sw})}")
rr = STATE.get("real_reorder")
if rr:
    print(f"raw reorder body         : {rr['body']!r}")
mm = STATE.get("metric_matrix") or {}
if mm:
    movers = [m for m, v in mm.items() if len({str(x) for x in v}) > 1]
    print(f"metrics moving with order: {movers or 'NONE'}")
d = STATE.get("discriminator")
if d:
    print(f"gauge on data change     : {fmt(d['before'])} -> {fmt(d['after'])}")
print()
print("close the session when finished:  tm1.logout()")